In [16]:
# Install
!pip install -q transformers accelerate

In [17]:
# Import
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [21]:
# Simple Evaluator Class

class SimpleBenchmarkEvaluator:

    def __init__(self, model_name="tiiuae/falcon-rw-1b"):
        print(f"Loading model: {model_name}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )

        self.model.eval()

    def evaluate_question(self, question, choices, correct_idx):
        scores = []

        for choice in choices:
            prompt = f"Question: {question}\nAnswer: {choice}"

            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                loss = outputs.loss.item()

            scores.append(-loss)  # higher is better

        pred_idx = scores.index(max(scores))
        return pred_idx, pred_idx == correct_idx

    def run(self, questions):
        correct = 0

        for i, q in enumerate(questions):
            pred, is_correct = self.evaluate_question(
                q["question"],
                q["choices"],
                q["correct_idx"]
            )

            if is_correct:
                correct += 1

            print(f"[{i+1}] {'yes' if is_correct else 'no'} Predicted: {q['choices'][pred]}")

        total = len(questions)
        acc = correct / total * 100


        print(f"Accuracy: {correct}/{total} = {acc:.2f}%")



In [22]:
# Sample question
questions = [
    {
        "question": "What is the chemical symbol for gold?",
        "choices": ["Ag", "Au", "Fe", "Cu"],
        "correct_idx": 1,
    },
    {
        "question": "Which planet is closest to the Sun?",
        "choices": ["Venus", "Earth", "Mercury", "Mars"],
        "correct_idx": 2,
    },
    {
        "question": "What is the derivative of x^2?",
        "choices": ["x", "2x", "x^2", "2x^2"],
        "correct_idx": 1,
    },
]


In [23]:
# Run evaluation
evaluator = SimpleBenchmarkEvaluator("tiiuae/falcon-rw-1b")
evaluator.run(questions)

Loading model: tiiuae/falcon-rw-1b


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

[1] yes Predicted: Au
[2] yes Predicted: Mercury
[3] no Predicted: x^2
Accuracy: 2/3 = 66.67%
